In [11]:
import pandas as pd
import os
import time
from sqlalchemy import create_engine
import logging

# Check if the logs directory exists, if not, create it
if not os.path.exists('logs'):
    os.makedirs('logs')

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename="logs/ingestion_db.logs",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s", 
    filemode="a"
)

engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, engine):
    '''this function will ingest the dataframe in the database table'''
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)

def load_raw_data():
    '''this function will load the csvs as dataframe and load in the database '''
    start = time.time()
    
    # Check if data directory exists to avoid errors
    if not os.path.exists('data'):
        logging.error("Data directory not found!")
        return

    for file in os.listdir('data'):
        if file.endswith('.csv'): # A cleaner way to check for CSVs
            file_path = os.path.join('data', file)
            df = pd.read_csv(file_path)
            
            # Record the start of ingestion for this specific file
            logging.info(f"Ingesting {file} in db")
            
            # file[:-4] removes '.csv' to use the filename as the table name
            ingest_db(df, file[:-4], engine)
            
    end = time.time()
    total_time = (end - start) / 60
   
    logging.info("Ingestion completed")
    logging.info(f"Total time taken: {total_time:.2f} minutes") # Rounded to 2 decimal places

if __name__ == "__main__":
    load_raw_data()

If your data ingestion script runs overnight and fails, you can't see the terminal output. The log file provides a "black box" record of exactly which file caused the crash.

(206529, 9)
(224489, 9)
(2372474, 16)
(12261, 9)
(4555004, 14)
(5543, 10)
